In [1]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

25/12/10 18:16:36 WARN Utils: Your hostname, Manojrams-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.219 instead (on interface en0)
25/12/10 18:16:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/manojrammopati/.ivy2/cache
The jars for the packages stored in: /Users/manojrammopati/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-azure added as a dependency
com.azure#azure-storage-blob added as a dependency
com.azure#azure-identity added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7748d3fd-3ba9-4752-9af7-61268e239a06;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/envs/spark_local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-azure;3.3.4 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.12.7 in central
	found org.slf4j#slf4j-api;1.7.36 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.5.2 in central
	found com.google.errorpron

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [2]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


# Cell 1 — Setup

In [3]:
# Cell 1 — Setup Spark, imports, paths

from pyspark.sql import functions as F
import importlib
import sys
from datetime import datetime
from pathlib import Path

# 1) Resolve project root (adjust if needed)
PROJECT_ROOT = Path("/Users/manojrammopati/Public/Projects/bupa_insurance_project")

# 2) Make sure utils_silver can be imported
silver_utils_dir = PROJECT_ROOT / "_02_Silver"
if str(silver_utils_dir) not in sys.path:
    sys.path.insert(0, str(silver_utils_dir))

import utils_silver
importlib.reload(utils_silver)

# 3) Build paths (reuse your config)
storage_account = "clientdatastorage"  # <== keep as you used elsewhere
paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account)

print("Silver paths:", sorted(paths_silver.keys()))
print("Gold   paths:", sorted(paths_gold.keys()))

# 4) Databases
DB_SILVER = "bupa_silver"
DB_GOLD   = "bupa_gold"


✅ utils_silver.py loaded
Silver paths: ['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Gold   paths: ['dq_monitoring', 'fact_claims', 'fact_members']


# Cell 2 — Helper to compute DQ stats for one table

In [4]:
# Cell 2 — Helper to profile a single table

def dq_profile_table(
    df,
    layer: str,
    table_name: str,
    key_cols=None,
    dq_flag_prefix: str = "dq_"
):
    """
    Compute simple DQ metrics for one Spark DataFrame.

    Returns: dict suitable for write_dq_snapshot()
    """
    if key_cols is None:
        key_cols = []

    row_count = df.count()

    # Count nulls in key columns
    key_nulls = 0
    for c in key_cols:
        if c in df.columns:
            key_nulls += df.filter(F.col(c).isNull()).count()

    # Any dq_* flags?
    dq_cols = [c for c in df.columns if c.startswith(dq_flag_prefix)]
    dq_bad_rows = 0
    if dq_cols:
        cond = None
        for c in dq_cols:
            c_bad = (F.col(c) == 0)
            cond = c_bad if cond is None else (cond | c_bad)
        dq_bad_rows = df.filter(cond).count()

    notes = f"keys={key_cols}, dq_cols={dq_cols}"

    return {
        "run_id": f"{layer}:{table_name}:{datetime.utcnow().isoformat(timespec='seconds')}",
        "layer": layer,
        "table_name": table_name,
        "row_count": row_count,
        "key_nulls": key_nulls,
        "dq_bad_rows": dq_bad_rows,
        "notes": notes,
    }


# Cell 3 — Collect DQ snapshots (Silver + Gold)

In [5]:
DB_SILVER = "bupa_silver"
DB_GOLD   = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_SILVER}.policies
    USING DELTA
    LOCATION 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/policies'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_SILVER}.members
    USING DELTA
    LOCATION 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/members'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_SILVER}.claims
    USING DELTA
    LOCATION 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/claims'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_SILVER}.providers
    USING DELTA
    LOCATION 'abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_claims
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_policies
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_members
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dim_channel
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_channel'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dim_product_line
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_product_line'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dim_member_segment
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_member_segment'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dim_region
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_region'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.dim_providers
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_providers'
""")





DataFrame[]

In [6]:
# Cell 3 — Compute DQ metrics for key tables

snapshots = []

# ---------- SILVER LAYER ----------

# Silver policies
silver_policies = spark.table(f"{DB_SILVER}.policies")
snapshots.append(
    dq_profile_table(
        df=silver_policies,
        layer="silver",
        table_name="policies",
        key_cols=["Policy_ID", "Customer_ID"]
    )
)

# Silver members
silver_members = spark.table(f"{DB_SILVER}.members")
snapshots.append(
    dq_profile_table(
        df=silver_members,
        layer="silver",
        table_name="members",
        key_cols=["Member_ID", "Policy_ID"]
    )
)

# Silver claims
silver_claims = spark.table(f"{DB_SILVER}.claims")
snapshots.append(
    dq_profile_table(
        df=silver_claims,
        layer="silver",
        table_name="claims",
        key_cols=["Claim_ID", "Member_Key", "Provider_ID"]
    )
)

# Silver providers
silver_providers = spark.table(f"{DB_SILVER}.providers")
snapshots.append(
    dq_profile_table(
        df=silver_providers,
        layer="silver",
        table_name="providers",
        key_cols=["Provider_ID"]
    )
)

# ---------- GOLD FACTS ----------

fact_claims  = spark.table(f"{DB_GOLD}.fact_claims")
fact_members = spark.table(f"{DB_GOLD}.fact_members")
fact_policies = spark.table(f"{DB_GOLD}.fact_policies")

snapshots.append(
    dq_profile_table(
        df=fact_claims,
        layer="gold",
        table_name="fact_claims",
        key_cols=["Claim_ID", "Member_Key", "Provider_ID"]
    )
)
snapshots.append(
    dq_profile_table(
        df=fact_members,
        layer="gold",
        table_name="fact_members",
        key_cols=["Member_ID", "Policy_ID"]
    )
)
snapshots.append(
    dq_profile_table(
        df=fact_policies,
        layer="gold",
        table_name="fact_policies",
        key_cols=["Policy_ID", "Customer_ID"]
    )
)

# ---------- GOLD DIMENSIONS ----------

dim_channel = spark.table(f"{DB_GOLD}.dim_channel")
dim_product_line = spark.table(f"{DB_GOLD}.dim_product_line")
dim_member_segment = spark.table(f"{DB_GOLD}.dim_member_segment")
dim_region = spark.table(f"{DB_GOLD}.dim_region")
dim_providers = spark.table(f"{DB_GOLD}.dim_providers")

snapshots.append(
    dq_profile_table(
        df=dim_channel,
        layer="gold",
        table_name="dim_channel",
        key_cols=["Channel_Code"]
    )
)

snapshots.append(
    dq_profile_table(
        df=dim_product_line,
        layer="gold",
        table_name="dim_product_line",
        key_cols=["Product_Line_Code"]
    )
)

snapshots.append(
    dq_profile_table(
        df=dim_member_segment,
        layer="gold",
        table_name="dim_member_segment",
        key_cols=["Member_Segment_Code"]
    )
)

snapshots.append(
    dq_profile_table(
        df=dim_region,
        layer="gold",
        table_name="dim_region",
        key_cols=["Region_Code"]
    )
)

snapshots.append(
    dq_profile_table(
        df=dim_providers,
        layer="gold",
        table_name="dim_providers",
        key_cols=["Provider_ID"]
    )
)

print("Number of DQ snapshot rows:", len(snapshots))
snapshots[:3]


25/12/10 18:16:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Number of DQ snapshot rows: 12


[{'run_id': 'silver:policies:2025-12-10T18:17:03',
  'layer': 'silver',
  'table_name': 'policies',
  'row_count': 381109,
  'key_nulls': 0,
  'dq_bad_rows': 66981,
  'notes': "keys=['Policy_ID', 'Customer_ID'], dq_cols=['dq_money_valid', 'dq_discount_valid', 'dq_renewal_valid']"},
 {'run_id': 'silver:members:2025-12-10T18:17:06',
  'layer': 'silver',
  'table_name': 'members',
  'row_count': 381109,
  'key_nulls': 0,
  'dq_bad_rows': 0,
  'notes': "keys=['Member_ID', 'Policy_ID'], dq_cols=['dq_age_valid', 'dq_bmi_valid']"},
 {'run_id': 'silver:claims:2025-12-10T18:17:09',
  'layer': 'silver',
  'table_name': 'claims',
  'row_count': 558211,
  'key_nulls': 39142,
  'dq_bad_rows': 0,
  'notes': "keys=['Claim_ID', 'Member_Key', 'Provider_ID'], dq_cols=['dq_money_valid', 'dq_date_valid']"}]

# Cell 4 — Write DQ monitoring snapshot (Delta table)

In [7]:
# Cell 4 — Persist snapshots into golddata/dq_monitoring

utils_silver.write_dq_snapshot(
    spark=spark,
    snapshot_rows=snapshots,
    paths_gold=paths_gold
)

# Register as a table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_GOLD}.dq_monitoring
USING DELTA
LOCATION '{paths_gold["dq_monitoring"]}'
""")

print(f"✅ DQ monitoring table available as {DB_GOLD}.dq_monitoring")


[DQ] Wrote 12 rows into dq_monitoring (abfss://golddata@clientdatastorage.dfs.core.windows.net/dq_monitoring)
✅ DQ monitoring table available as bupa_gold.dq_monitoring


# Cell 5 — Quick sanity check & dashboard-friendly view

In [8]:
# Cell 5 — Inspect latest DQ run

dq_df = spark.read.format("delta").load(paths_gold["dq_monitoring"])

print("Latest DQ snapshot:")
latest_ts = dq_df.agg(F.max("run_ts").alias("max_ts")).collect()[0]["max_ts"]

latest = dq_df.filter(F.col("run_ts") == latest_ts)

latest.orderBy("layer", "table_name").show(100, truncate=False)

# Simple pivot: row_count vs dq_bad_rows by table
summary = (
    latest
      .select("layer", "table_name", "row_count", "key_nulls", "dq_bad_rows")
      .orderBy("layer", "table_name")
)

summary.show(100, truncate=False)


Latest DQ snapshot:


+-------------------------------------------+--------------------------+------+------------------+---------+---------+-----------+------------------------------------------------------------------------------------------------------+
|run_id                                     |run_ts                    |layer |table_name        |row_count|key_nulls|dq_bad_rows|notes                                                                                                 |
+-------------------------------------------+--------------------------+------+------------------+---------+---------+-----------+------------------------------------------------------------------------------------------------------+
|gold:dim_channel:2025-12-10T18:17:22       |2025-12-10 18:17:27.460501|gold  |dim_channel       |4        |0        |0          |keys=['Channel_Code'], dq_cols=[]                                                                     |
|gold:dim_member_segment:2025-12-10T18:17:25|2025-12-10 18:17:27

# 1️⃣ bupa_gold.data_quality_run_summary

In [9]:
# Create / refresh a summary view aggregated by run

spark.sql("""
CREATE OR REPLACE VIEW bupa_gold.data_quality_run_summary AS
SELECT
    run_ts,
    layer,
    COUNT(DISTINCT table_name)          AS table_count,
    SUM(row_count)                      AS total_rows,
    SUM(key_nulls)                      AS total_key_nulls,
    SUM(dq_bad_rows)                    AS total_dq_bad_rows
FROM bupa_gold.dq_monitoring
GROUP BY run_ts, layer
""")

print("✅ View bupa_gold.data_quality_run_summary created.")
spark.table("bupa_gold.data_quality_run_summary").orderBy("run_ts", "layer").show(truncate=False)


✅ View bupa_gold.data_quality_run_summary created.


+--------------------------+------+-----------+----------+---------------+-----------------+
|run_ts                    |layer |table_count|total_rows|total_key_nulls|total_dq_bad_rows|
+--------------------------+------+-----------+----------+---------------+-----------------+
|2025-12-09 23:41:43.956415|gold  |8          |1327324   |39142          |66981            |
|2025-12-09 23:41:43.956415|silver|4          |1325822   |39142          |66981            |
|2025-12-10 18:17:27.460501|gold  |8          |1327324   |39142          |66981            |
|2025-12-10 18:17:27.460501|silver|4          |1325822   |39142          |66981            |
+--------------------------+------+-----------+----------+---------------+-----------------+



Now you have:

bupa_gold.dq_monitoring – detailed per table, per run.

bupa_gold.data_quality_run_summary – run-level summary for dashboards.

That fully covers the “data quality run summary” idea.